In [42]:
import pandas as pd
import numpy as np

In [43]:
df = pd.read_csv("../data/raw/returning_customer.csv")

# Dropping cols

In [44]:
df = df.drop(columns = ["Order_ID", "Customer_ID", "Total_Amount"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      22049 non-null  object 
 1   Age                       22049 non-null  int64  
 2   Gender                    22049 non-null  object 
 3   City                      22049 non-null  object 
 4   Product_Category          22049 non-null  object 
 5   Unit_Price                22049 non-null  float64
 6   Quantity                  22049 non-null  int64  
 7   Discount_Amount           22049 non-null  float64
 8   Payment_Method            22049 non-null  object 
 9   Device_Type               22049 non-null  object 
 10  Session_Duration_Minutes  22049 non-null  int64  
 11  Pages_Viewed              22049 non-null  int64  
 12  Is_Returning_Customer     22049 non-null  bool   
 13  Delivery_Time_Days        22049 non-null  int64  
 14  Custom

# Changing date format

In [45]:
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")
df['Date']

0       2023-05-29
1       2023-10-12
2       2023-12-05
3       2023-05-11
4       2023-06-16
           ...    
22044   2024-03-26
22045   2024-03-26
22046   2024-03-26
22047   2024-03-26
22048   2024-03-26
Name: Date, Length: 22049, dtype: datetime64[ns]

In [46]:
df["Is_Returning_Customer"] = df["Is_Returning_Customer"].astype("int64")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      22049 non-null  datetime64[ns]
 1   Age                       22049 non-null  int64         
 2   Gender                    22049 non-null  object        
 3   City                      22049 non-null  object        
 4   Product_Category          22049 non-null  object        
 5   Unit_Price                22049 non-null  float64       
 6   Quantity                  22049 non-null  int64         
 7   Discount_Amount           22049 non-null  float64       
 8   Payment_Method            22049 non-null  object        
 9   Device_Type               22049 non-null  object        
 10  Session_Duration_Minutes  22049 non-null  int64         
 11  Pages_Viewed              22049 non-null  int64         
 12  Is_Returning_Custo

# Feature Engineering

In [47]:
df['engagement_score'] = df['Session_Duration_Minutes'] * df['Pages_Viewed']
df['Date'] = pd.to_datetime(df['Date'])
df['day_of_week'] = df['Date'].dt.dayofweek 
df['month'] = df['Date'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['fast_delivery'] = (df['Delivery_Time_Days'] <= 3).astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      22049 non-null  datetime64[ns]
 1   Age                       22049 non-null  int64         
 2   Gender                    22049 non-null  object        
 3   City                      22049 non-null  object        
 4   Product_Category          22049 non-null  object        
 5   Unit_Price                22049 non-null  float64       
 6   Quantity                  22049 non-null  int64         
 7   Discount_Amount           22049 non-null  float64       
 8   Payment_Method            22049 non-null  object        
 9   Device_Type               22049 non-null  object        
 10  Session_Duration_Minutes  22049 non-null  int64         
 11  Pages_Viewed              22049 non-null  int64         
 12  Is_Returning_Custo

# Encoding non-numericals

In [48]:
cols_list = [x for x in df.columns if df[x].dtype in ["object", "bool"]]
hot_encoding = []

for col in cols_list:
    if df[col].nunique()<=2:
        binary_cols = df[col].unique()
        df[col] = df[col].map({binary_cols[0]: 1, binary_cols[1]: 0})
    else:
        hot_encoding.append(col)

df = pd.get_dummies(df, columns=hot_encoding, drop_first=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             22049 non-null  datetime64[ns]
 1   Age                              22049 non-null  int64         
 2   Unit_Price                       22049 non-null  float64       
 3   Quantity                         22049 non-null  int64         
 4   Discount_Amount                  22049 non-null  float64       
 5   Session_Duration_Minutes         22049 non-null  int64         
 6   Pages_Viewed                     22049 non-null  int64         
 7   Is_Returning_Customer            22049 non-null  int64         
 8   Delivery_Time_Days               22049 non-null  int64         
 9   Customer_Rating                  22049 non-null  int64         
 10  engagement_score                 22049 non-null  int64    

# Feature selection

In [49]:
df = df.drop("Date", axis=1)
X = df.drop("Is_Returning_Customer", axis=1)
y = df["Is_Returning_Customer"]

In [50]:
#variance

def variance_find(X, threshold):
    var = np.var(X, axis=0)
    worth_cols = [i for i, j in enumerate(var) if j>threshold]
    return X.iloc[:, worth_cols]

X_var = variance_find(X, 0.05)
X_var.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             22049 non-null  int64  
 1   Unit_Price                      22049 non-null  float64
 2   Quantity                        22049 non-null  int64  
 3   Discount_Amount                 22049 non-null  float64
 4   Session_Duration_Minutes        22049 non-null  int64  
 5   Pages_Viewed                    22049 non-null  int64  
 6   Delivery_Time_Days              22049 non-null  int64  
 7   Customer_Rating                 22049 non-null  int64  
 8   engagement_score                22049 non-null  int64  
 9   day_of_week                     22049 non-null  int32  
 10  month                           22049 non-null  int32  
 11  is_weekend                      22049 non-null  int64  
 12  fast_delivery                   

In [51]:
# Multicollinearity

corr_matrix = X_var.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col]>0.8)]
X_multicorr = X_var.drop(to_drop, axis=1)

X_multicorr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22049 entries, 0 to 22048
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             22049 non-null  int64  
 1   Unit_Price                      22049 non-null  float64
 2   Quantity                        22049 non-null  int64  
 3   Discount_Amount                 22049 non-null  float64
 4   Session_Duration_Minutes        22049 non-null  int64  
 5   Pages_Viewed                    22049 non-null  int64  
 6   Delivery_Time_Days              22049 non-null  int64  
 7   Customer_Rating                 22049 non-null  int64  
 8   engagement_score                22049 non-null  int64  
 9   day_of_week                     22049 non-null  int32  
 10  month                           22049 non-null  int32  
 11  is_weekend                      22049 non-null  int64  
 12  fast_delivery                   

# Train and Test split

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =train_test_split(
    X_multicorr,
    y,
    random_state=42,
    train_size = 0.8,
    test_size = 0.2,
    shuffle = True,
    stratify = y
)
print("After split:")
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


After split:
(17639, 33)
(4410, 33)
(17639,)
(4410,)


In [52]:
# def IQR_method(df):
#     df = df.copy().astype(float)
#     logged = np.log1p(df)
#     bounds = {}
#     df_cleaned = logged.copy()
#     for col in df:
#         Q1 = logged[col].quantile(0.25)
#         Q3 = logged[col].quantile(0.75)
#         IQR = Q3 - Q1
#         lower = Q1 - 1.5 * IQR
#         upper = Q3 + 1.5 * IQR
#         bounds[col] = {"lower": lower, "upper": upper}
#         df_cleaned[col] = logged[col].clip(lower, upper)
#     return df_cleaned, bounds

# X_train_scaled, bounds = IQR_method(X_train)

# X_test_scaled = X_test.copy().astype(float)
# X_test_scaled = np.log1p(X_test_scaled)
# for col in X_test_scaled.columns:
#     if col not in bounds:
#         continue
#     lower = bounds[col]["lower"]
#     upper = bounds[col]["upper"]
#     X_test_scaled[col] = X_test_scaled[col].clip(lower, upper)

def fit_scale(df):
    df = df.copy().astype(float)
    logged = np.log1p(df)
    mu_sigma = {}
    df_cleaned = logged.copy()
    for col in logged.columns:
        mu = logged[col].mean()
        sigma = logged[col].std()
        mu_sigma[col]= {"mu": mu, "sigma": sigma}
        df_cleaned[col] = (logged[col]-mu)/sigma
    return df_cleaned, mu_sigma
        
X_train_scaled, mu_sigma = fit_scale(X_train)

X_test_scaled = X_test.copy().astype(float)
X_test_scaled = np.log1p(X_test_scaled)
for col in X_test_scaled.columns:
    mu = mu_sigma[col]["mu"]
    sigma = mu_sigma[col]["sigma"]
    X_test_scaled[col] = (X_test_scaled[col] - mu) / sigma

print(len(X_train_scaled), len(X_test_scaled))

17639 4410


# SMOTE Data

In [55]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

print(len(X_train_resampled), len(y_train_resampled))

28846 28846


# Saving dataset

In [56]:
X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
print(f"Length of X_train_resampled: {len(X_train_resampled)}")
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
print(f"Length of X_test_scaled: {len(X_test_scaled)}")
y_train_resampled.to_csv('../data/processed/y_train.csv', index=False)
print(f"Length of y_train_resampled: {len(y_train_resampled)}")
y_test.to_csv('../data/processed/y_test.csv', index=False)
print(f"Length of y_test: {len(y_test)}")

import json
with open('../data/processed/mu_sigma.json', 'w') as f:
    json.dump(mu_sigma, f, indent=2)

print("Bundle Saved!")
print("Files saved!")

Length of X_train_resampled: 28846
Length of X_test_scaled: 4410
Length of y_train_resampled: 28846
Length of y_test: 4410
Bundle Saved!
Files saved!
